In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Sequential computations in JAX: `jax.lax.scan`
Many computations are inherently sequential: each step consumes the result of the one before it. Examples include integrating an equation of motion or more generaly evaluating any recurrence of the form $y_{n+1}=f(y_n,x_n)$.
The natural way to write this is a Python for loop, but inside a JAX transformation that loop is unrolled and we've already seen how it can come back to bite us.

We now introduce `jax.lax.scan` as a solution to this class of problems ([link to the documentation](https://docs.jax.dev/en/latest/_autosummary/jax.lax.scan.html), read it!).
You write the body of a single step as a pure function, and scan marches it along an input sequence, or simply for a fixed number of steps, compiling that body only once into one rolled loop.
Each step threads a carry, the state passed from one iteration to the next, and emits an output that scan stacks into an array: the carry is the running state you fold forward and the stacked outputs are the trace you keep.

## Simple example - Cumulative sum
As a first simple illustration, let us implement a cumulative sum function using `scan`.
In terms of serie, for a given vector $u x_n$, its cumulative sum is given by
$$u_{n+1} = u_n + x_n,$$
$$u_0 = x_0.$$

In [ ]:
xs = jnp.array([3., 1., 4., 1., 5., 9.])
@jax.jit
def cumulative_sum(xs, x_init=0.):
    def add(carry, x):
        # Pure function. Only output the sum of the current value with its preceding cumulated one.
        new = carry + x
        return new, new
    total, running = jax.lax.scan(add, x_init, xs)
    return total, running

Let us compare both implementations:

In [ ]:
print(cumulative_sum(xs))
print(jnp.sum(xs), jnp.cumsum(xs))

They're perfectly identical, what a surprise.

## Exercice - Geometric series & compound interest

Let us suppose we invest $u_0$ with a fixed annual interest rate $r$.
At year $n$, the total accumulated value becomes

$$u_{n+1} = (1+r)u_n.$$

As a geometric serie, the total accumulated value at year $n$ can be directly computed as

$$u_n = (1+r)^nu_0.$$

Using `jax.lax.scan`, compute the value of the total accumulated value up to year $n$ and compare it to the direct calculation.

In [ ]:
def compound_interest(x_init, rate, years):
    def step(x, _):
        return (1.+rate)*x, x         # carry evolves; record value before update
    final, history = jax.lax.scan(step, x_init, length=years)
    return final, history

> _**Note:**_ In this example, we set `xs` to `None`. Instead, the loop count is define by `length`. In that case, `xs` is built as a tuple of `None`.

In [ ]:
print(compound_interest(100., 0.02, 10))
print((1+0.02)**9*100)

## Exercice - Euler integrator

Now that we master financial mathematics, let us focus on numerically solving Ordinal Differential Equations (ODE) of the form

$$\frac{\mathrm{d}f}{\mathrm{d}t} = A(x, f),$$

with $A(x, f)$ an arbitrary continuous and first order differentiable function (first order ODE).
In this example, we will solve this equation using the forward Euler method.

Forward Euler advances the solution one step at a time:
$$y_{n+1} = y_n + h A(x_n, y_n).$$

TODO: compléter un peu ce block.

In [ ]:
def euler_integrate_scan(A, y_0, x_0, x_N, h):
    x = jnp.linspace(x_0, x_N, ((x_N-x_0)/h).astype(int))
    def step(y, x_i):
        return y + h * A(x_i, y), y

    return x, jax.lax.scan(step, y_0, x)[1]

> _**Exercice:**_ Rewrite the Euler integration function this time specifying the abscisse vector on which to solve the ODE (that is, infer $h$).

In [ ]:
# Uncomment this to get the solution
# %load solutions/euler_integrate_scan2.py

Lets apply our Euler integration scheme on a function of the form

$$A(t, f) = f.$$

The solution to the differential equation is then simply the exponential function.

In [ ]:
def A(t, f):
    return f

In [ ]:
x_N = jnp.array(3.)
h = 0.001
x, y_euler = euler_integrate_scan(A, 1., 0., x_N, h)
y_func = jnp.exp(x)

In [ ]:
plt.subplots(nrows=2, ncols=1, figsize=(6., 4.), layout='constrained', gridspec_kw={'height_ratios': [1., 0.4]}, sharex=True)
plt.subplot(2, 1, 1)
plt.plot(x, y_euler, label="Euler")
plt.plot(x, y_func, label="Analytical")
plt.ylabel("$y$")
plt.legend()
plt.grid()
plt.subplot(2, 1, 2)
plt.plot(x, y_euler-y_func)
plt.ylabel("Residuals")
plt.grid()
plt.xlabel("$x$")
plt.show()

> _**Exercice:**_
Now that you have a nice integrator, study its convergence against the analytical solution to the ODE (compute its residuals).
Try to use `vmap` to sweep over step sizes. Will it work? Explain why or why not.

In [ ]:
# Uncomment this to get the solution!
# %load solutions/euler_error.py